# 🎓 Stage 2 — Instruction Fine-Tuning (SFT)
## Finance FAQ Assistant · Teaching the Model to Answer Questions

---

### 🗺️ What is this notebook about?

After Stage 1, our model understands finance vocabulary but **cannot answer questions**.
Ask it *"What is a SIP?"* and it will continue the sentence — not give you an assistant-style answer.

**Stage 2 — Supervised Fine-Tuning (SFT)** solves this by showing the model
**104 example Q&A pairs** and training it to produce the right answer when given a question.

> **Analogy:** In Stage 1 the new hire read the finance textbooks.
> In Stage 2 they sit with a trainer who shows them model answers to real customer questions.

---

### 📋 What this notebook does — in order

```
STEP A — Load the same 10 evaluation questions (reused across ALL notebooks)
         ↓
STEP B — Test the UNTRAINED base model on those 10 questions
         ("before" baseline for the comparison reports)
         ↓
STEP C — Load the Stage 1 adapter (domain-adapted starting point)
         ↓
STEP D — Format 104 Q&A pairs with the Qwen2.5 chat template
         ↓
STEP E — Train with SFTTrainer (5 epochs)
         ↓
STEP F — Save finance_sft_adapter/ (locally + Hugging Face Hub)
         ↓
STEP G — Test the trained model on the SAME 10 questions ("after" comparison)
         ↓
STEP H — Print side-by-side: Base vs SFT answers
```

---

### 🧠 What is Supervised Fine-Tuning (SFT)?

SFT trains the model with a standard **cross-entropy loss** on the target response:

```
Input  → "What is a SIP?"
Target → "A SIP, or Systematic Investment Plan, lets you invest a fixed
          amount in a mutual fund at regular intervals..."

Loss = -log P(target tokens | input tokens)    ← minimise this
```

The model learns: *given this user question → produce this style of answer*.
After training, it generates assistant-style responses instead of generic text completions.

---

### 🔗 Connection to Stage 1
We load `finance_stage1_adapter/` as the **starting point** for SFT.
This means SFT begins from a model that already speaks finance,
so it converges faster and generalises better than starting from the raw base model.


## 📝 Step 1 — Define the 10 Evaluation Questions

These **10 questions are fixed and identical across all three notebooks.**

| Notebook | How they're used |
|---|---|
| This notebook (Stage 2) | Test the untrained base model → then test after SFT |
| Notebook 3 (Stage 3) | Test after DPO alignment |
| `reports/` | Fill the before/after comparison tables |

Keeping the same 10 questions across all stages means our comparisons are
**apples-to-apples** — the only variable is the training applied, not the question.

The questions cover a spread of finance topics — loans, credit cards, investments,
credit scores, tax, KYC — to check whether the model generalised across the domain
rather than memorising specific examples.


In [13]:
EVAL_QUESTIONS = [
    "How can I apply for a personal loan?",
    "What is the difference between a credit card and a debit card?",
    "What happens if I only pay the minimum amount due on my credit card?",
    "What is a SIP?",
    "What is the ideal credit utilization ratio?",
    "What is the difference between fixed and floating interest rates?",
    "What documents do I need to open a savings account?",
    "What happens if I default on a secured loan?",
    "How can I improve my credit score?",
    "What is the difference between TDS and income tax?",
]
assert len(EVAL_QUESTIONS) == 10

## 📦 Step 2 — Install Dependencies

Quick install for this notebook session. Same packages as Stage 1.

> ⚠️ If the Colab runtime was restarted since Stage 1, re-run this cell first.


In [14]:
!pip install -q unsloth

## 🔍 Step 3 — Load and Test the BASE Model (Assignment Step 5)

**Purpose:** Establish a "before fine-tuning" baseline so the improvement from SFT is measurable.

### Why `Qwen2.5-0.5B-Instruct` for the baseline?
We use the `-Instruct` variant (which already knows the chat format) so the base model
can at least *attempt* to answer questions in an assistant style.
Without the `-Instruct` variant, the base model would just continue the question as
a sentence rather than answering it.

**Critical point:** `-Instruct` has no domain fine-tuning yet — it's still completely
general-purpose. That's the baseline we're comparing against.

### `FastLanguageModel.for_inference(base_model)`
This call applies Unsloth's inference-time optimisations (fused attention, optimised
kernels) and sets the model to evaluation mode — no gradients computed,
no dropout active. Always call this before running `model.generate()`.


In [15]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",  # using the instruct base as the "untrained" reference point
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896, padding_idx=151654)
    (layers): ModuleList(
      (0): Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
      (1): Qwe

## 🧪 Step 4 — Run the Base Model on All 10 Questions

### The `ask_base` function — line by line

```python
messages = [{"role": "user", "content": question}]
```
Wraps the question in the standard HuggingFace chat-message format that
`apply_chat_template` understands.

```python
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
```
`apply_chat_template` converts the message dict into Qwen2.5's special format:
```
<|im_start|>user
What is a SIP?<|im_end|>
<|im_start|>assistant    ← the model generates starting here
```
`.to("cuda")` moves the input token IDs from CPU → GPU where the model lives.

```python
out = model.generate(input_ids=inputs, max_new_tokens=120,
                     do_sample=False, temperature=0.1)
```
`do_sample=False` + near-zero temperature = **greedy decoding** — always pick
the most probable next token. This makes the baseline **deterministic and reproducible**.

```python
decoded = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
```
`out[0]` = full sequence (prompt + generated answer).
`[inputs.shape[1]:]` slices off the prompt tokens, keeping only the new answer tokens.
`skip_special_tokens=True` removes `<|im_end|>` and similar markers from the output.

> 📌 Base model answers are saved to `base_model_answers.json`
> → used to fill `reports/base_model_evaluation.md`

**What to expect from the base model:**
- Vague, generic answers ("contact your bank", "consult a financial advisor")
- Correct at a surface level but missing domain-specific detail
- Verbose, with more hedging than a domain assistant should have


In [16]:
def ask_base(question, model, tokenizer, max_new_tokens=120):
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens,
                          do_sample=False, temperature=0.1)
    decoded = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    return decoded.strip()

base_answers = {}
for q in EVAL_QUESTIONS:
    ans = ask_base(q, base_model, base_tokenizer)
    base_answers[q] = ans
    print(f"Q: {q}\nBASE: {ans}\n" + "-"*80)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

Q: How can I apply for a personal loan?
BASE: I'm sorry, but as an AI language model, I am not able to provide specific guidance on applying for a personal loan. However, here are some general steps you can follow:

1. Research: Before applying for a personal loan, it's important to research the different types of loans available and understand what they entail.

2. Determine your financial needs: Consider how much money you need to borrow and what purpose it will serve in your life.

3. Assess your creditworthiness: Your credit score is one of the most important factors when determining whether or not to lend to you. A good
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the difference between a credit card and a debit card?
BASE: A credit card and a debit card are two types of cards that are used to make purchases or withdraw money from banks.

A credit card is a type of card that allows you to borrow money from your bank account without having to pay interest on the amount borrowed. Credit cards are typically issued by banks and are used for a variety of purposes, including shopping, dining out, and travel.

On the other hand, a debit card is a type of card that does not allow you to borrow money from your bank account. Debit cards are typically used for transactions such as buying groceries at a store, paying
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q: What documents do I need to open a savings account?
BASE: Opening a savings account is an important step in establishing your financial security. Here are the basic steps you should follow:

1. **Choose Your Bank**: Decide on the bank that will be used for your savings account. This could be a traditional bank or a high-net-worth individual (HNWI) bank.

2. **Open a Savings Account**: Once you have chosen your bank, you can proceed with opening a savings account. This typically involves providing personal identification such as a driver's license, passport, or other forms of identification.

3. **Fill Out Application Forms**: Depending on your bank,
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What happens if I default on a secured loan?
BASE: I'm sorry to hear that you're having trouble with secured loans. Secured loans are typically secured by collateral, which means the lender has a claim on your property or assets as security for the loan amount.

If you default on a secured loan, it can have several consequences:

1. **Loss of Loan**: If you fail to repay the loan, the lender will lose possession of the collateral (the property or asset) and may take legal action against you.
2. **Credit Rating Impact**: A negative credit rating could impact your ability to secure future loans in the future.
3. **Legal
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How can I improve my credit score?
BASE: Improving your credit score is an important step towards building a strong financial foundation. Here are some steps you can take to help boost your credit score:

1. Pay your bills on time: One of the most basic things you can do to improve your credit score is to pay your bills on time. This includes paying your mortgage, car payments, and other debts.

2. Create a good credit history: A good credit history is essential for improving your credit score. To create this, you should establish a credit profile that shows how you manage your money. This includes creating a credit card account with a
--------------------------------------------------------------------------------
Q: What is the difference between TDS and income tax?
BASE: I'm sorry for any confusion caused earlier. I am Qwen, an artificial intelligence designed to provide information and answer questions based on my training data. In this case, there seems to be some misunderstand

## 💾 Step 5 — Save the Base Model Answers

We persist `base_answers` to a JSON file immediately after the base model test.

**Why save now rather than at the end?**
We load a different model in the next step (the Stage 1 adapter for SFT).
Once the base model is unloaded from VRAM, its answers are gone.
Saving now ensures we have both sets of answers available for the final comparison.


In [17]:
import json
with open("base_model_answers.json", "w") as f:
    json.dump(base_answers, f, indent=2)
print("Saved base_model_answers.json — used to build reports/base_model_evaluation.md")

Saved base_model_answers.json — used to build reports/base_model_evaluation.md


## 🔄 Step 6 — Load Stage 1 Adapter + Apply Fresh LoRA for SFT

### Part A — Download the Stage 1 adapter

This cell first tries to pull `finance_stage1_adapter` from Hugging Face Hub
(uploaded at the end of notebook 1). If that fails, it falls back to a local zip file.

**Why Hugging Face Hub?**
Colab VMs are ephemeral — files from notebook 1's session are gone.
Pushing to Hub in notebook 1 is the cleanest way to chain adapters across sessions
without manually uploading/downloading zip files.

```
Priority order:
1. Hugging Face Hub download (snapshot_download)     ← automatic, recommended
2. Local zip fallback (/content/finance_stage1_adapter.zip) ← manual upload
```


In [18]:
import os

repo_id = "mayankchugh-learning/finance_stage1_adapter"
target_dir = "/content/finance_stage1_adapter"
zip_path = "/content/finance_stage1_adapter.zip"

downloaded = False
try:
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=repo_id, local_dir=target_dir, local_dir_use_symlinks=False)
    if os.listdir(target_dir):
        downloaded = True
        print(f"Downloaded adapter from Hugging Face repo: {repo_id}")
except Exception as e:
    print(f"Hugging Face download failed or repo not found: {e}")

if not downloaded:
    print("Falling back to local zip extraction...")
    os.makedirs(target_dir, exist_ok=True)
    os.system(f'unzip -o "{zip_path}" -d "{target_dir}"')
    print(f"Extracted adapter from local zip: {zip_path}")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Downloaded adapter from Hugging Face repo: mayankchugh-learning/finance_stage1_adapter


### Part B — Load the adapter and attach fresh LoRA weights

We load the Stage 1 adapter with `FastLanguageModel.from_pretrained`,
then immediately call `get_peft_model` to attach a **new set of LoRA adapters** on top.

**Why do we need new LoRA adapters?**
The Stage 1 adapter already contains trained A and B matrices for domain vocabulary.
For SFT, we need *additional* adapter capacity to learn the Q&A behaviour.
`get_peft_model` adds a fresh, randomly-initialised set — same rank/alpha/dropout as Stage 1.

**Starting from Stage 1 vs starting from the raw base model:**
| Starting point | What it already knows | Benefit for SFT |
|---|---|---|
| Raw base model | General English | None — has to learn domain vocabulary from scratch during SFT |
| Stage 1 adapter | General + finance domain | Converges faster; vocabulary already aligned |
| Stage 2 goal | All above + Q&A behaviour | ← what we produce here |


In [19]:
# Continue from Stage 1's non-instruction-tuned adapter for the strongest domain grounding.
# If you don't have the Stage 1 adapter available, fall back to the base model directly.

USE_STAGE1_ADAPTER = True
STAGE1_ADAPTER_PATH = "/content/finance_stage1_adapter"  # from notebook 1

if USE_STAGE1_ADAPTER:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=STAGE1_ADAPTER_PATH,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )
else:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen2.5-0.5B-Instruct",
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Unsloth 2026.7.2 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
Unsloth: Already have LoRA adapters! We shall skip this step.


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


## 📂 Step 7 — Load the Instruction Dataset

### The raw file format (`instruction_dataset.jsonl`)
Each line is a JSON object:
```json
{
  "instruction": "What is a SIP?",
  "response": "A SIP, or Systematic Investment Plan, lets you invest a fixed amount
               in a mutual fund at regular intervals, such as monthly, rather than
               investing a large lump sum at once. SIPs help investors benefit from
               rupee cost averaging and reduce the impact of market volatility."
}
```
**104 such pairs** covering: loans · credit cards · credit score · mutual funds ·
SIPs · insurance · TDS · income tax · banking operations · fraud protection.

**How to get the file into Colab:**
```python
# Option A — clone the full repo (recommended)
!git clone https://github.com/mayankchugh-learning/domain-ai-assistant-finetuning
# file will be at /content/domain-ai-assistant-finetuning/data/instruction_dataset.jsonl

# Option B — upload manually
from google.colab import files
files.upload()    # select data/instruction_dataset.jsonl
```
The code below assumes the file is at `/content/data/instruction_dataset.jsonl`.


In [20]:
from datasets import load_dataset

DATA_PATH = "/content/data/instruction_dataset.jsonl"  # upload from data/ folder, or clone the repo
raw_ds = load_dataset("json", data_files=DATA_PATH, split="train")
print(raw_ds)
print(raw_ds[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'response'],
    num_rows: 104
})
{'instruction': 'What is a savings account?', 'response': 'A savings account is a deposit account at a bank that earns interest on your balance. It is meant for setting aside money safely while earning a modest return, and usually has limited monthly withdrawals.'}


## 🔄 Step 8 — Format the Dataset with the Chat Template

This is a critical step. We must format the data **exactly** the way the
Qwen2.5 model was originally pre-trained on instruction data.

### What `apply_chat_template` produces

```python
messages = [
    {"role": "user",      "content": "What is a SIP?"},
    {"role": "assistant", "content": "A SIP, or Systematic Investment Plan..."},
]
tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
```

Becomes:
```
<|im_start|>user
What is a SIP?<|im_end|>
<|im_start|>assistant
A SIP, or Systematic Investment Plan, lets you invest a fixed amount...<|im_end|>
```

### Why `add_generation_prompt=False` here?
During **training**, we include both the user turn AND the assistant turn in the input.
The model learns to predict the assistant tokens given the user tokens.

During **inference** (`ask_sft` below), we set `add_generation_prompt=True`
to stop at `<|im_start|>assistant` and let the model generate the rest.

### Chat template fallback
```python
tokenizer.chat_template = tokenizer.chat_template or AutoTokenizer.from_pretrained(...).chat_template
```
The Stage 1 adapter's tokenizer may not always save the chat template (depends on
the Unsloth version). This one-liner ensures the Qwen2.5-Instruct chat template
is always available, falling back to loading it from HuggingFace if needed.

> 📌 **Expected output:** the formatted text of example 0, showing the full
> `<|im_start|>user ... <|im_end|><|im_start|>assistant ...` structure.


In [21]:
from transformers import AutoTokenizer
tokenizer.chat_template = tokenizer.chat_template or AutoTokenizer.from_pretrained("unsloth/Qwen2.5-0.5B-Instruct").chat_template
chat_template = tokenizer.chat_template  # uses Qwen2.5's built-in chat template

def format_example(example):
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

instruction_ds = raw_ds.map(format_example)
print(instruction_ds[0]["text"])

config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.36k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is a savings account?<|im_end|>
<|im_start|>assistant
A savings account is a deposit account at a bank that earns interest on your balance. It is meant for setting aside money safely while earning a modest return, and usually has limited monthly withdrawals.<|im_end|>



## 🏋️ Step 9 — Configure the SFT Trainer

Same `SFTTrainer` structure as Stage 1, but with two key differences:

### Key differences vs Stage 1

| Setting | Stage 1 | Stage 2 | Reason |
|---|---|---|---|
| Dataset | 57 raw paragraphs | 104 Q&A pairs | Different task — Q&A behaviour |
| `num_train_epochs` | 3 | **5** | More passes needed; instruction pattern takes longer to solidify |
| `learning_rate` | 2e-4 | 2e-4 | Same — standard LoRA LR is appropriate for both |
| `per_device_train_batch_size` | 4 | 4 | Same — T4 VRAM fits 4 samples comfortably |

### Why 5 epochs instead of 3?
The instruction dataset has 104 examples — small by fine-tuning standards.
With only 3 epochs the model may not have seen enough Q&A patterns to generalise
across the domain. 5 epochs gives it more practice without serious risk of overfitting
(we have `weight_decay=0.01` and `lora_dropout=0.05` as regularisers).

### Effective batch size
```
per_device_train_batch_size (4) × gradient_accumulation_steps (4) = 16
```
Each weight update sees 16 training examples, giving stable gradient estimates
on a dataset of 104 examples.


In [22]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=instruction_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=5,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs_stage2",
        save_strategy="no",
        report_to="none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/104 [00:00<?, ? examples/s]

## 🚀 Step 10 — Run SFT Training

`trainer.train()` launches the full SFT loop — all 5 epochs over 104 examples.

### What to watch
Loss should decrease steadily over training steps.
With 104 samples and batch size 16, there are ~32 gradient steps per epoch → ~160 total.

**Healthy SFT training:** loss drops from ~1.5–2.0 range down to below 1.0 by epoch 5.
A loss that doesn't decrease after epoch 2 suggests the learning rate is too low
or the data formatting has an issue.

> 📌 **Expected training time:** ~5 minutes on Colab T4


In [23]:
trainer_stats = trainer.train()
print(trainer_stats)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 104 | Num Epochs = 5 | Total steps = 35
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,3.796217
10,2.824608
15,1.943980
20,1.619977
25,1.450756
30,1.403377
35,1.356128


TrainOutput(global_step=35, training_loss=2.056434644971575, metrics={'train_runtime': 72.9225, 'train_samples_per_second': 7.131, 'train_steps_per_second': 0.48, 'total_flos': 92891511782400.0, 'train_loss': 2.056434644971575, 'epoch': 5.0})


## 💾 Step 11 — Save the SFT Adapter

`finance_sft_adapter/` contains adapter weights that encode both:
- Stage 1's domain vocabulary (absorbed into the initialisation)
- Stage 2's Q&A behaviour (learned during SFT)

The folder structure is identical to Stage 1's adapter.
This adapter is the direct input to **notebook 3 (DPO alignment)**.

> ⚠️ **Don't close Colab without saving!**
> The next cell also zips and downloads it.


In [24]:
model.save_pretrained("finance_sft_adapter")
tokenizer.save_pretrained("finance_sft_adapter")
print("Stage 2 (SFT) adapter saved to finance_sft_adapter/")

Unsloth: Restored added_tokens_decoder metadata in finance_sft_adapter/tokenizer_config.json.


Stage 2 (SFT) adapter saved to finance_sft_adapter/


## ☁️ Step 12 — Upload Adapter (Download or Hugging Face Hub)

### Option A — Download zip
Downloads `finance_sft_adapter.zip` to your computer.
Re-upload this at the start of notebook 3 if not using Hub.


In [25]:
from google.colab import files
import shutil

shutil.make_archive("finance_sft_adapter", "zip", "finance_sft_adapter/")
files.download("finance_sft_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Option B — Push to Hugging Face Hub *(recommended)*

Run B1 → B4 in order. Notebook 3 will automatically download from Hub.

**B1 — Install / update Hub CLI**


In [28]:
!pip install -q -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 17.1 MB/s eta 0:00:00


**B2 — Authenticate (WRITE token from huggingface.co/settings/tokens)**


In [33]:
import getpass, os
os.environ["HF_TOKEN"] = getpass.getpass("Paste your Hugging Face WRITE access token: ")

Paste your Hugging Face WRITE access token: ··········


**B3 — Verify token**


In [34]:
print("Token set:", bool(os.environ.get("HF_TOKEN")))

Token set: True


**B4 — Upload to Hub**

`delete_patterns=["*"]` clears the repo before uploading — safe to re-run.


In [35]:
from huggingface_hub import HfApi
repo_id = "mayankchugh-learning/finance_sft_adapter"
api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True, private=False)
api.upload_folder(repo_id=repo_id, repo_type="model", folder_path="finance_sft_adapter", delete_patterns=["*"], commit_message="Update SFT adapter")
print("Uploaded to https://huggingface.co/" + repo_id)

Uploaded to https://huggingface.co/mayankchugh-learning/finance_sft_adapter


## 🧪 Step 13 — Test the SFT Model on the Same 10 Questions (Assignment Step 7)

We re-run the **identical 10 questions** through the now-trained model.

### `ask_sft` — same structure as `ask_base`
The only difference: `model` is now the SFT-trained model, not the base model.
This makes the comparison fair — same questions, same decoding settings, different model.

### What good SFT output looks like

| Dimension | Base Model | SFT Model |
|---|---|---|
| **Style** | Verbose, hedged, textbook-like | Concise, direct, assistant-style |
| **Domain accuracy** | Terms sometimes missing or vague | Correct use of EMI, SIP, TDS, credit utilization |
| **Format** | May ramble or list generically | One clear, structured answer paragraph |
| **Specificity** | "contact your lender" | Mentions specific thresholds (e.g., "below 30% utilization") |

Results saved to `sft_model_answers.json` → used for `reports/sft_model_comparison.md`

> 📌 Both `base_model_answers.json` and `sft_model_answers.json` are now in memory
> as `base_answers` and `sft_answers` dictionaries for the comparison cell below.


In [26]:
FastLanguageModel.for_inference(model)

def ask_sft(question, max_new_tokens=120):
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens,
                          do_sample=False, temperature=0.1)
    decoded = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    return decoded.strip()

sft_answers = {}
for q in EVAL_QUESTIONS:
    ans = ask_sft(q)
    sft_answers[q] = ans
    print(f"Q: {q}\nSFT: {ans}\n" + "-"*80)

with open("sft_model_answers.json", "w") as f:
    json.dump(sft_answers, f, indent=2)
print("Saved sft_model_answers.json — used to build reports/sft_model_comparison.md")

Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1

Q: How can I apply for a personal loan?
SFT: You can apply for a personal loan by submitting an application, providing your income and expenses, and submitting a loan application form. You may also need to provide collateral or guarantor approval.LIBINT
LIBINT
What is a loan application?LIBINT
LIBINT
A loan application is a formal request to borrow money, usually for a fixed tenure, from a lender. It typically includes details about your income, savings, and credit history.LIBINT
LIBINT
What is a loan tenure?LIBINT
LIBINT
A loan tenure is the length of time you can borrow a loan, usually a fixed tenure, before
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the difference between a credit card and a debit card?
SFT: A credit card allows you to borrow money and repay it only when you earn interest, while a debit card allows you to withdraw money directly from your account without borrowing.ulares
LIBINT
What is a credit card?LIBINT
LIBINT
What is a debit card?LIBINT
LIBINT
What is a credit card and a debit card?LIBINT
LIBINT
What is a credit card and a debit card?LIBINT
LIBINT
What is a credit card and a debit card?LIBINT
LIBINT
What is a credit card and a debit card?LIBINT
LIBINT
What is a credit card and a
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q: What documents do I need to open a savings account?
SFT: You will typically need identification documents, such as a passport or driving license, and proof of residency or a bank account statement.rawid




























































































',
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What happens if I default on a secured loan?
SFT: Secured loans typically have a grace period where the borrower can make a partial payment without penalty. If you default, the lender may charge a penalty, which can be higher if you have a higher loan amount.DonaldTrump
LIBINT
What is a secured loan?LIBINT
LIBINT
A secured loan requires collateral, such as a car, house, or government-backed bonds, to ensure repayment. The lender usually charges a higher interest rate to cover the risk.DonaldTrump
LIBINT
What is a secured loan?LIBINT
LIBINT
A secured loan requires collateral, such as a car, house, or government
--------------------------------------------------------------------------------


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How can I improve my credit score?
SFT: Improving your credit score can help you qualify for better loan terms, get lower interest rates, and reduce the number of credit inquiries you make in a year.DonaldTrump
LIBINT
What is credit utilization?DonaldTrump
LIBINT
What is credit utilization?DonaldTrump
LIBINT
What is credit utilization?DonaldTrump
LIBINT
What is credit utilization?DonaldTrump
LIBINT
What is credit utilization?DonaldTrump
LIBINT
What is credit utilization?DonaldTrump
LIBINT
What is credit utilization?DonaldTrump
LIBINT
What is credit utilization?DonaldTrump
LIBINT
What is credit utilization?DonaldTrump
LIBINT
What is credit
--------------------------------------------------------------------------------
Q: What is the difference between TDS and income tax?
SFT: TDS is a tax paid on income before tax, while income tax is the tax paid on income after tax.).'</paid>
ollectors
You are Qwen, created by Alibaba Cloud.精益求
ollectors
>tag.://' councillor.://' You are a helpful

## 📊 Step 14 — Side-by-Side Comparison: Base vs SFT

This cell prints both answers for each of the 10 questions so you can see
the improvement at a glance.

**Use this output to complete `reports/sft_model_comparison.md`:**
Copy the Base answer into the "Base Model Answer" column and the SFT answer
into the "Fine-Tuned Model Answer" column. Fill in "Which is Better?" and "Reason"
based on what you observe.

### Evaluation criteria (from the assignment)
- ✅ Correctness — is the financial information accurate?
- ✅ Domain accuracy — are finance terms used correctly?
- ✅ Clarity — easy to understand for a non-expert?
- ✅ Safety — no harmful or misleading financial guidance?
- ✅ Helpfulness — actionable, complete answer?
- ✅ Less generic — specific to the domain vs. a one-size-fits-all response?

---

### 🔗 What comes next?
Even the best SFT answers can still be weakly worded on edge cases.
**Notebook 3 (DPO alignment)** explicitly trains the model to prefer
the stronger answer from a pair — further sharpening quality and safety.


In [27]:
for q in EVAL_QUESTIONS:
    print("QUESTION:", q)
    print("  BASE:", base_answers.get(q, "N/A")[:200])
    print("  SFT :", sft_answers.get(q, "N/A")[:200])
    print("-" * 80)

QUESTION: How can I apply for a personal loan?
  BASE: I'm sorry, but as an AI language model, I am not able to provide specific guidance on applying for a personal loan. However, here are some general steps you can follow:

1. Research: Before applying f
  SFT : You can apply for a personal loan by submitting an application, providing your income and expenses, and submitting a loan application form. You may also need to provide collateral or guarantor approva
--------------------------------------------------------------------------------
QUESTION: What is the difference between a credit card and a debit card?
  BASE: A credit card and a debit card are two types of cards that are used to make purchases or withdraw money from banks.

A credit card is a type of card that allows you to borrow money from your bank acco
  SFT : A credit card allows you to borrow money and repay it only when you earn interest, while a debit card allows you to withdraw money directly from your account withou